# Stage C — 25M-class calibration and matched training
Default: a 50k-base numerical-stability calibration of the 25.27M-parameter model. Only change `STUDY_PHASE` to `primary` after this calibration and Notebook 02 capacity gate both pass.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='8579b06b641e442191277a61ecfd3644cf1593c4'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
STUDY_PHASE='calibration'  # Change to 'primary' only after the two gates pass.
BLOCK_COUNT=11
D_MODEL=512
NUM_HEADS=8
PERSISTENT_TOKENS=4
MEMORY_DEPTH=1
HORIZON=3
LEARNING_RATE=3e-5
GRADIENT_CLIP_NORM=0.5
VALIDATION_STREAMS=4
PHASES={
    'calibration': {'pilot_name':'c9_25m_model_calibration_50k','valid_base_budget':50_000,'checkpoint_every':25,'evidence_tier':'engineering'},
    'primary': {'pilot_name':'c10_25m_model_matched_25mbp','valid_base_budget':25_000_000,'checkpoint_every':1_000,'evidence_tier':'confirmatory'},
}
if STUDY_PHASE not in PHASES: raise ValueError('STUDY_PHASE must be calibration or primary')
phase=PHASES[STUDY_PHASE]
PILOT_NAME=phase['pilot_name']
VALID_BASE_BUDGET=phase['valid_base_budget']
CHECKPOINT_EVERY=phase['checkpoint_every']
EVIDENCE_TIER=phase['evidence_tier']
RUN_IDS={
    'calibration': {mode:'calibration_50k' for mode in ('adaptive','reference','frozen_memory','no_memory')},
    'primary': {'adaptive':'adaptive_discovery_25m','reference':'control_reference_equal_budget','frozen_memory':'control_frozen_equal_budget','no_memory':'control_none_equal_budget'},
}[STUDY_PHASE]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess,sys
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
selection_path=Path(DRIVE_ROOT)/'runs'/'c1_tokenizers_cpu'/'tokenizer_selection.json'
selection=json.loads(selection_path.read_text(encoding='utf-8'))
selected=selection.get('selected_tokenizer')
if not isinstance(selected,str) or not selected: raise ValueError('tokenizer_selection.json has no selected_tokenizer')
dataset=Path(DRIVE_ROOT)/'stage_c_dataset'/'ordered_streams'/selected
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError(f'Run Notebook 00b first; missing {dataset}/token_stream_manifest.json')
DATASET_DIR=str(dataset)
print('Resolved Handoff 00b dataset:',DATASET_DIR)
root=f'{DRIVE_ROOT}/runs/{PILOT_NAME}'
PROTOCOL=repo/'studies'/'stage_c_ecoli_escherichia_medium_25m_v1'/'protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study'/'stage_c_ecoli_escherichia_medium_25m_v1'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',root,'--label','hardware_preflight','--repo',str(repo),'--','seqtrainer-titans-stage-c-hardware-preflight','--require','A100'],check=True)

In [ ]:
def show_failure(run_dir, label):
    failure=Path(run_dir)/'FAILED.txt'
    log=Path(run_dir)/'logs'/f'{label}.log'
    if failure.exists(): print(failure.read_text(encoding='utf-8',errors='replace'))
    if log.exists():
        print(f'--- tail of {log} ---')
        print(log.read_text(encoding='utf-8',errors='replace')[-12000:])

for mode in ['adaptive','reference','frozen_memory','no_memory']:
    run_dir=f'{root}/{mode}'
    label=f'train_{mode}'
    command=['seqtrainer-titans-stage-c-train','--dataset-dir',DATASET_DIR,'--run-dir',run_dir,'--memory-mode',mode,'--horizon',str(HORIZON),'--batch-size','1','--max-valid-bases',str(VALID_BASE_BUDGET),'--checkpoint-every',str(CHECKPOINT_EVERY),'--learning-rate',str(LEARNING_RATE),'--gradient-clip-norm',str(GRADIENT_CLIP_NORM),'--validation-streams',str(VALIDATION_STREAMS),'--activation','float32','--block-count',str(BLOCK_COUNT),'--d-model',str(D_MODEL),'--num-heads',str(NUM_HEADS),'--persistent-tokens',str(PERSISTENT_TOKENS),'--memory-depth',str(MEMORY_DEPTH),'--protocol',str(PROTOCOL),'--run-id',RUN_IDS[mode]]
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',run_dir,'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        show_failure(run_dir,label)
        raise
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',RUN_IDS[mode],'--evidence-tier',EVIDENCE_TIER,'--artifact',run_dir],check=True)
print('SHARE THIS DIRECTORY:',root)
print('Live status during training: <run>/<mode>/LIVE_STATUS.json')